# 02 - Chain of Thought

CoT prompting, self-consistency, tree of thoughts, automatic CoT, and zero-shot CoT.


In [ ]:
import numpy as np
import re
import math
import matplotlib.pyplot as plt
from collections import Counter
import random
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. Zero-Shot CoT


In [ ]:
# Zero-shot CoT: Add 'Let's think step by step'
def zero_shot_cot(question):
    return f'Question: {question}\n\nLet us think step by step.\n\nAnswer:'

questions = [
    'If a train travels 60 km in 1 hour, how far does it travel in 2.5 hours?',
    'What is 15% of 200?',
    'A rectangle has length 8 and width 5. What is its area?'
]
for q in questions:
    print(f'\n{zero_shot_cot(q)}\n')


## 2. Manual CoT Examples


In [ ]:
# Manual CoT: Provide reasoning examples
cot_examples = [
    ('What is 17 + 28?', 'First, add 7 + 8 = 15. Write down 5, carry 1. Then 1 + 1 + 2 = 4. So the answer is 45.'),
    ('If 3 apples cost $6, how much do 5 apples cost?', 'First, find the cost per apple: $6 / 3 = $2 per apple. Then multiply by 5: $2 * 5 = $10.'),
    ('What is 25% of 80?', '25% means 1/4. So 80 / 4 = 20.')
]

def manual_cot_prompt(question, examples):
    prompt = 'Solve the following problems by thinking step by step.\n\n'
    for q, reasoning in examples:
        prompt += f'Q: {q}\nA: {reasoning}\n\n'
    prompt += f'Q: {question}\nA:'
    return prompt

new_question = 'If 4 workers can build a wall in 6 days, how long will 8 workers take?'
print(manual_cot_prompt(new_question, cot_examples))


## 3. Self-Consistency


In [ ]:
# Self-consistency: Generate multiple reasoning paths and vote
def generate_reasoning_paths(question, n_paths=5):
    # Simulate different reasoning paths
    paths = []
    for i in range(n_paths):
        # Different approaches
        if i == 0:
            path = f'Path {i+1}: Direct calculation approach... Result: 42'
        elif i == 1:
            path = f'Path {i+1}: Algebraic method... Result: 42'
        elif i == 2:
            path = f'Path {i+1}: Estimation approach... Result: 41'
        elif i == 3:
            path = f'Path {i+1}: Pattern matching... Result: 42'
        else:
            path = f'Path {i+1}: Dimensional analysis... Result: 42'
        paths.append(path)
    return paths

def self_consistency_vote(paths):
    # Extract answers and vote
    answers = [re.search(r'Result: (\d+)', p).group(1) for p in paths if re.search(r'Result: (\d+)', p)]
    vote_counts = Counter(answers)
    winner = vote_counts.most_common(1)[0]
    return winner[0], vote_counts

paths = generate_reasoning_paths('Sample question', n_paths=5)
for p in paths:
    print(p)
winner, counts = self_consistency_vote(paths)
print(f'\nConsensus answer: {winner} (votes: {dict(counts)})')


## 4. Tree of Thoughts


In [ ]:
# Tree of Thoughts: Explore multiple reasoning branches
class ThoughtNode:
    def __init__(self, thought, parent=None, score=0):
        self.thought = thought
        self.parent = parent
        self.children = []
        self.score = score
    def add_child(self, thought, score):
        child = ThoughtNode(thought, self, score)
        self.children.append(child)
        return child
    def get_path(self):
        path = []
        node = self
        while node:
            path.append(node.thought)
            node = node.parent
        return list(reversed(path))

# Build a reasoning tree
root = ThoughtNode('Problem: Find the best ML algorithm for text classification')

# Level 1
n1 = root.add_child('Consider dataset size', 0.8)
n2 = root.add_child('Consider model complexity', 0.7)
n3 = root.add_child('Consider training time', 0.6)

# Level 2 from n1
n1_1 = n1.add_child('Small dataset -> Naive Bayes', 0.9)
n1_2 = n1.add_child('Large dataset -> Deep Learning', 0.85)

# Level 2 from n2
n2_1 = n2.add_child('Simple task -> Logistic Regression', 0.8)
n2_2 = n2.add_child('Complex task -> Transformer', 0.9)

# Find best path
def find_best_path(node):
    if not node.children:
        return node.get_path(), node.score
    best_path, best_score = None, -1
    for child in node.children:
        path, score = find_best_path(child)
        if score > best_score:
            best_score = score
            best_path = path
    return best_path, best_score

best_path, best_score = find_best_path(root)
print('Best reasoning path:')
for step in best_path:
    print(f'  -> {step}')
print(f'Score: {best_score}')


## 5. Automatic CoT Generation


In [ ]:
# Auto-CoT: Generate examples automatically
def auto_cot_generate(demonstrations, k=3):
    # Cluster questions by complexity and select diverse examples
    clusters = {'easy': [], 'medium': [], 'hard': []}
    for q, a in demonstrations:
        word_count = len(q.split())
        if word_count < 10:
            clusters['easy'].append((q, a))
        elif word_count < 20:
            clusters['medium'].append((q, a))
        else:
            clusters['hard'].append((q, a))
    

    selected = []
    for cluster in clusters.values():
        if cluster:
            selected.append(random.choice(cluster))
    

    return selected[:k]

demos = [
    ('What is 2+2?', '2 + 2 = 4'),
    ('What is 15 * 4?', '15 * 4 = 60. Break it down: 10*4=40, 5*4=20, 40+20=60.'),
    ('If a car travels at 60 mph for 2 hours, how far?', 'Distance = speed * time = 60 * 2 = 120 miles.'),
    ('What is the area of a circle with radius 3?', 'Area = pi * r^2 = 3.14159 * 9 = 28.27.'),
    ('A store offers 20% off on a $50 item. What is the final price?', 'Discount = 50 * 0.20 = 10. Final price = 50 - 10 = $40.')
]

auto_examples = auto_cot_generate(demos, k=3)
print('Auto-selected CoT examples:')
for q, a in auto_examples:
    print(f'  Q: {q}')
    print(f'  A: {a}\n')


## 6. CoT Effectiveness Visualization


In [ ]:
methods = ['Direct', 'Zero-Shot CoT', 'Manual CoT', 'Self-Consistency', 'Tree of Thoughts', 'Auto-CoT']
math_accuracy = [35, 58, 72, 78, 81, 75]
commonsense_accuracy = [42, 65, 74, 80, 77, 73]

x = np.arange(len(methods))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, math_accuracy, width, label='Math Reasoning', color='steelblue')
plt.bar(x + width/2, commonsense_accuracy, width, label='Commonsense Reasoning', color='coral')
plt.xlabel('Method')
plt.ylabel('Accuracy (%)')
plt.title('Chain of Thought Methods Comparison')
plt.xticks(x, methods, rotation=15, ha='right')
plt.legend()
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


## 7. Exercises


In [ ]:
# Exercise 1: Implement step-by-step verification
# Exercise 2: Add backtracking in Tree of Thoughts
# Exercise 3: Implement ensemble CoT (combine multiple methods)
# Exercise 4: Build a CoT prompt optimizer
print('Exercises listed!')
